# Day 55 — Export: Delivering Data Like a Professional
**Month 3 | Week 5 | CSV · Excel · Formatting · Multi-Sheet · Client-Ready Output**

---

> **Real-world framing:**
>
> You've done the analysis. The client now says:
> *"Send me the file."*
>
> A junior analyst exports raw DataFrame output — messy column names, no formatting,
> everything in one sheet, no context.
>
> A professional analyst delivers a **client-ready file**: clean headers, formatted numbers,
> colour-coded sheets, summary on top, data below, frozen panes, auto-fitted columns.
>
> The analysis is identical. The perceived quality is not.
> Export is the last mile — and it's where trust is built or lost.

---

**Skills used today:** Pandas basics, GroupBy, Feature Engineering, Apply/Lambda (all prior days)  
**New today:** `to_csv()` · `to_excel()` with `ExcelWriter` · `openpyxl` formatting ·
multi-sheet workbooks · column width · number formats · freeze panes · named styles

**Total: 80 pts + 10★ bonus**

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify)

Same ShopEase dataset (200 records, seed=7). Run once. Work on copies only.

In [1]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS CELL ──────────────────────────────
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side,
                              numbers as xl_numbers)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=7)

n = 200
regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
segments   = ['Regular', 'Premium', 'VIP']
statuses   = ['Delivered', 'Returned', 'Pending']

raw = pd.DataFrame({
    'order_id'    : [f'ORD{1000+i}' for i in range(n)],
    'order_date'  : pd.date_range('2024-01-01', periods=n, freq='2D'),
    'region'      : rng.choice(regions,    n, p=[0.30, 0.25, 0.25, 0.20]),
    'category'    : rng.choice(categories, n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'product'     : [f'P{rng.integers(100,150):03d}' for _ in range(n)],
    'units'       : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(50, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n, p=[0.4,0.2,0.2,0.1,0.1]),
    'segment'     : rng.choice(segments,   n, p=[0.50, 0.30, 0.20]),
    'status'      : rng.choice(statuses,   n, p=[0.70, 0.21, 0.09]),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] *
                  (1 - raw['discount_pct']/100)).round(2)
raw['month']   = raw['order_date'].dt.to_period('M')
raw['quarter'] = raw['order_date'].dt.to_period('Q')

# Pre-built derived columns (from Days 52–54) — do not re-derive
def revenue_tier(rev):
    if rev >= 5000:   return 'High Value'
    elif rev >= 2000: return 'Mid Value'
    else:             return 'Low Value'

df = raw.copy()
df['rev_tier']       = df['revenue'].apply(revenue_tier)
df['action_code']    = df['status'].map({'Delivered':'CLOSE','Returned':'REFUND','Pending':'FOLLOW-UP'})
df['rev_per_unit']   = (df['revenue'] / df['units']).round(2)

print(f"Dataset: {df.shape}")
print(df[['order_id','region','category','revenue','rev_tier','action_code']].head(5))
# ── END RAW DATA ──────────────────────────────────────────────────────────


Dataset: (200, 16)
  order_id region     category  revenue    rev_tier action_code
0  ORD1000   East       Sports  6584.18  High Value       CLOSE
1  ORD1001   West         Home  3119.92   Mid Value      REFUND
2  ORD1002   East         Home  1027.70   Low Value       CLOSE
3  ORD1003  North  Electronics  1290.02   Low Value      REFUND
4  ORD1004  South         Home   886.95   Low Value      REFUND


---
## 📖 Section 2 — Concept Notes

---

### 1. CSV Export — `to_csv()`

The simplest export. Use for data pipelines, database ingestion, sharing with non-Excel users.

```python
df.to_csv('output.csv', index=False)           # always set index=False
df.to_csv('output.csv', index=False, encoding='utf-8-sig')   # for Excel-safe UTF-8 (₹ symbol)
```

**Key parameters:**
| Parameter | Use |
|-----------|-----|
| `index=False` | Never export the row number index — it's meaningless to clients |
| `encoding='utf-8-sig'` | Prevents garbled characters when opened in Excel on Windows |
| `columns=[...]` | Export only specific columns |
| `float_format='%.2f'` | Force 2 decimal places in the CSV text |

---

### 2. Basic Excel Export — `to_excel()`

```python
df.to_excel('report.xlsx', index=False, sheet_name='Orders')
```

For **multiple sheets**, use `ExcelWriter` as a context manager:

```python
with pd.ExcelWriter('report.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    detail_df.to_excel(writer, sheet_name='Raw Data', index=False)
    pivot_df.to_excel(writer, sheet_name='Pivot', index=True)
```

---

### 3. openpyxl Formatting — the Professional Layer

After writing data, you access the workbook via `writer.book` and apply formatting:

```python
with pd.ExcelWriter('report.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Sheet1', index=False)
    wb = writer.book
    ws = wb['Sheet1']

    # Header formatting
    header_fill = PatternFill('solid', fgColor='1F3864')   # dark blue
    header_font = Font(bold=True, color='FFFFFF', name='Arial', size=11)

    for cell in ws[1]:                          # row 1 = header
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
```

---

### 4. Number Formats

```python
from openpyxl.styles import numbers

# Currency
cell.number_format = '₹#,##0.00'       # ₹1,23,456.78
cell.number_format = '#,##0.00'         # 1,23,456.78 (no symbol)

# Percentage
cell.number_format = '0.00%'            # 12.34%

# Integer with comma
cell.number_format = '#,##0'            # 1,23,456
```

Apply to entire columns:
```python
for row in ws.iter_rows(min_row=2, min_col=5, max_col=5):   # column E
    for cell in row:
        cell.number_format = '#,##0.00'
```

---

### 5. Column Widths, Freeze Panes, Alternating Rows

```python
# Auto-fit column width based on max content length
for col in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

# Freeze top row (header stays visible when scrolling)
ws.freeze_panes = 'A2'

# Alternating row shading (zebra stripe)
light_fill = PatternFill('solid', fgColor='EBF0F8')
for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
    if i % 2 == 0:
        for cell in row:
            cell.fill = light_fill
```

---

### Common Mistakes & Fixes

| Mistake | Fix |
|---------|-----|
| `index=True` in export | Always `index=False` unless index is meaningful |
| Formatting lost on re-open | Use `openpyxl` engine, not `xlsxwriter` for reading back |
| Number stored as text in Excel | Set `number_format` on cells, don't export pre-formatted strings |
| `writer.save()` called manually | Use `with` context manager — it saves automatically |
| Hardcoded column letters (e.g., `'E'`) | Use `get_column_letter(col_index)` for maintainability |

---

### Interview One-liner
> "I always export analysis in two layers — the data in clean CSV for pipeline consumption,
> and a formatted Excel workbook with a summary sheet, formatted headers, number formats,
> and frozen panes for the client presentation. The analysis is the same;
> the formatted file is what builds trust in a client relationship."

---


---
## ✏️ Section 3 — Practice Tasks

Work only in this section. Never modify raw data.
Total = **80 pts + 10★ bonus**.

---

### Task A — CSV Export (15 pts)

**Business question:** The data engineering team needs a clean CSV of all Delivered orders
for database ingestion. They want only specific columns, UTF-8 encoded, no index.

#### A1 — Filtered CSV Export (8 pts)
Filter `df` to only `status == 'Delivered'` orders.  
Select these columns only: `order_id`, `order_date`, `region`, `category`,
`units`, `unit_price`, `revenue`, `rev_tier`.  
Export to `'delivered_orders.csv'` with:
- `index=False`
- `encoding='utf-8-sig'`
- `float_format='%.2f'`

After exporting, **read it back** with `pd.read_csv()` and print:
- Shape of the re-imported file
- First 5 rows


In [3]:
# A1 — Filtered CSV Export (8 pts)
# Step 1: Filter df to Delivered orders only
# Step 2: Select 8 specified columns
# Step 3: Export with to_csv() — index=False, utf-8-sig, float_format
# Step 4: Read back with pd.read_csv() and print shape + first 5 rows
delivered = df[df['status'] == 'Delivered'][
    ['order_id', 'order_date', 'region', 'category', 'units', 'unit_price', 'revenue', 'rev_tier']
].copy()
delivered.to_csv('ak_delivered_orders.csv', index = False, encoding = 'utf-8-sig', float_format='%.2f')
reimport = pd.read_csv('ak_delivered_orders.csv')
print(f"Shape: {reimport.shape}")
print(reimport.head(5).to_string(index = False))


Shape: (139, 8)
order_id order_date region category  units  unit_price  revenue   rev_tier
 ORD1000 2024-01-01   East   Sports     19      433.17  6584.18 High Value
 ORD1002 2024-01-05   East     Home      6      201.51  1027.70  Low Value
 ORD1006 2024-01-13  North   Sports      4      186.76   597.63  Low Value
 ORD1007 2024-01-15   West    Books      9      251.38  2262.42  Mid Value
 ORD1008 2024-01-17   East   Sports     16      467.60  7107.52 High Value


#### A2 — Export Verification (7 pts)
On the re-imported CSV DataFrame:
1. Confirm column count matches what you exported (print column list)
2. Confirm no index column was exported (check columns — should NOT contain `'Unnamed: 0'`)
3. Print the value counts of `rev_tier` in the re-imported file
4. Write one NRA sentence: what does the Delivered-only tier distribution tell the client?


In [4]:
# A2 — Export Verification (7 pts)
# 1. Print list of columns in re-imported CSV
# 2. Check 'Unnamed: 0' is not present — print True/False
# 3. Print value_counts of rev_tier from re-imported file
# No NRA here — write it in the markdown cell below
print("Columns:", reimport.columns.tolist())
print("No index col:", 'Unnamed: 0' not in reimport.columns)
print(reimport['rev_tier'].value_counts())


Columns: ['order_id', 'order_date', 'region', 'category', 'units', 'unit_price', 'revenue', 'rev_tier']
No index col: True
rev_tier
Low Value     77
Mid Value     45
High Value    17
Name: count, dtype: int64


<!-- A2 NRA — WRITE HERE -->

**N: Among the 139 delivered orders, 55.4% are Low Value (77) and only 12.2% are High Value (17).**

**R: This shows that most completed sales generate modest revenue, while high‑value transactions are rare even among successfully fulfilled orders.**

**A: The client should create post‑delivery up‑sell campaigns targeting Mid Value customers to convert more of them into High Value repeat buyers.**


---

### Task B — Basic Multi-Sheet Excel Export (20 pts)

**Business question:** The client's operations team wants a single Excel file with
three separate sheets: a region summary, a category summary, and the full order list.

#### B1 — Build the Three DataFrames (8 pts)
Create these three DataFrames from `df`:

**`region_summary`** — grouped by `region`:
- `total_revenue` (sum), `order_count` (count of order_id), `avg_revenue` (mean revenue)
- All rounded to 2 dp. Sort by `total_revenue` descending.

**`category_summary`** — grouped by `category`:
- `total_revenue` (sum), `order_count`, `avg_revenue`, `high_value_orders` (count where rev_tier == 'High Value')
- Sort by `total_revenue` descending.

**`order_detail`** — full `df` with columns:
`order_id`, `order_date`, `region`, `category`, `units`, `unit_price`,
`discount_pct`, `revenue`, `rev_tier`, `action_code`

Print all three DataFrames.


In [5]:
# B1 — Build Three DataFrames (8 pts)
# region_summary: groupby region → total_revenue, order_count, avg_revenue
# category_summary: groupby category → total_revenue, order_count, avg_revenue, high_value_orders
# order_detail: full df, 10 selected columns
# Print all three
region_summary = df.groupby('region').agg(
    total_revenue = ('revenue', 'sum'),
    order_count   = ('order_id', 'count'),
    avg_revenue   = ('revenue', 'mean')
).round(2).sort_values('total_revenue', ascending = False).reset_index()

category_summary = df.groupby('category').agg(
    total_revenue        = ('revenue', 'sum'),
    order_count          = ('order_id', 'count'),
    avg_revenue          = ('revenue', 'mean'),
    high_value_orders    = ('rev_tier', lambda x: (x=='High Value').sum())
).round(2).sort_values('total_revenue', ascending=False).reset_index()

order_detail = df[['order_id', 'order_date', 'region', 'category', 'units', 'unit_price',
                   'discount_pct', 'revenue', 'rev_tier', 'action_code']].copy()

print("=== region_summary ==="); print(region_summary.to_string(index = False))
print("\n=== category_summary ==="); print(category_summary.to_string(index = False))
print(f"\norder_detail shape: {order_detail.shape}")


=== region_summary ===
region  total_revenue  order_count  avg_revenue
  East      136318.29           47      2900.39
 North      132736.10           57      2328.70
 South      122708.43           52      2359.78
  West       97069.65           44      2206.13

=== category_summary ===
   category  total_revenue  order_count  avg_revenue  high_value_orders
Electronics      143021.83           62      2306.80                  7
   Clothing      123304.83           44      2802.38                  7
       Home       93885.39           35      2682.44                  3
     Sports       85764.71           42      2042.02                  4
      Books       42855.71           17      2520.92                  3

order_detail shape: (200, 10)


#### B2 — Write Multi-Sheet Excel File (12 pts)
Using `pd.ExcelWriter` as a context manager, write all three DataFrames to
`'shopease_report.xlsx'`:
- Sheet 1: `'Region Summary'` — `region_summary`, `index=False`
- Sheet 2: `'Category Summary'` — `category_summary`, `index=False`
- Sheet 3: `'Order Detail'` — `order_detail`, `index=False`

After writing, **read each sheet back** using `pd.read_excel()` with `sheet_name=` parameter
and print the shape of each to confirm all three sheets exported correctly.


In [7]:
# B2 — Multi-Sheet Excel Export (12 pts)
# Use: with pd.ExcelWriter('shopease_report.xlsx', engine='openpyxl') as writer:
#          df1.to_excel(writer, sheet_name='Region Summary', index=False)
#          ... (all 3 sheets)
# Then read back each sheet and print shape
with pd.ExcelWriter('ak_shopease_report.xlsx', engine = 'openpyxl') as writer:
    region_summary.to_excel(writer,   sheet_name='Region Summary',   index = False)
    category_summary.to_excel(writer, sheet_name='Category Summary', index = False)
    order_detail.to_excel(writer,     sheet_name='Order Detail',     index = False)

for sheet in ['Region Summary', 'Category Summary', 'Order Detail']:
    shape = pd.read_excel('ak_shopease_report.xlsx', sheet_name = sheet).shape
    print(f"{sheet}: {shape}")


Region Summary: (4, 4)
Category Summary: (5, 5)
Order Detail: (200, 10)


---

### Task C — Formatted Excel Export (30 pts)

**Business question:** The client's CEO wants a presentation-ready Excel file.
Same data as Task B but with professional formatting applied.

You will format `'shopease_formatted.xlsx'` — one sheet: `'Executive Summary'`
containing `region_summary` (from B1).

#### C1 — Write + Apply Header Formatting (10 pts)
Write `region_summary` to sheet `'Executive Summary'`.  
Then open `writer.book` and apply to **row 1 (header)**:
- Background: dark blue `#1F3864`
- Font: bold, white, Arial, size 11
- Alignment: center
- Print confirmation: `"Header formatted — rows: X, cols: Y"`


#### C2 — Number Formats + Alternating Rows (10 pts)
Continuing inside the same `ExcelWriter` block (or re-open with `load_workbook`):

Apply to the data rows (row 2 onwards):
- `total_revenue` column → number format `'#,##0.00'`
- `avg_revenue` column → number format `'#,##0.00'`
- `order_count` column → number format `'#,##0'`
- Alternating row fill: even rows → light blue `'EBF0F8'`, odd rows → no fill

Print: `"Number formats and row shading applied."`

> **Tip:** Find column index by `region_summary.columns.tolist().index('total_revenue') + 1`


#### C3 — Column Widths + Freeze Panes (10 pts)
Still inside the same block:
- Auto-fit all column widths (max content length + 4 padding)
- Freeze pane at `'A2'` so the header stays visible when scrolling

After the `with` block closes, **read back** `'shopease_formatted.xlsx'`
sheet `'Executive Summary'` and print the DataFrame to confirm data integrity
(formatting is visual — data must still be correct).


In [10]:
# C1 — Header Formatting (10 pts)
# with pd.ExcelWriter('shopease_formatted.xlsx', engine='openpyxl') as writer:
#     region_summary.to_excel(writer, sheet_name='Executive Summary', index=False)
#     wb = writer.book
#     ws = wb['Executive Summary']
#     Apply dark blue header to row 1
#     Print row/col count confirmation
# C2 — Number Formats + Alternating Rows (10 pts)
# Best approach: do C1 and C2 together in ONE ExcelWriter block
# Apply header (C1) + number formats + alternating rows (C2) before the 'with' block closes
# Structure:
# with pd.ExcelWriter(...) as writer:
#     region_summary.to_excel(...)
#     wb = writer.book; ws = wb['Executive Summary']
#     [C1 header formatting]
#     [C2 number formats on specific columns]
#     [C2 alternating row fill]
# C3 — Column Widths + Freeze Panes (10 pts)
# Add inside the same ExcelWriter block from C2:
#   - Auto-fit column widths using get_column_letter + column_dimensions
#   - ws.freeze_panes = 'A2'
# After block: read back and print to confirm data is intact
# IMPORTANT: C1 + C2 + C3 should all be in ONE single ExcelWriter block
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

with pd.ExcelWriter('ak_shopease_formatted.xlsx', engine='openpyxl')as writer:
    region_summary.to_excel(writer, sheet_name = 'Executive Summary', index = False)
    wb = writer.book
    ws = wb['Executive Summary']

    # C1 - Header
    hdr_fill = PatternFill('solid', fgColor = '1F3864')
    hdr_font = Font(bold = True, color = 'FFFFFF', name = 'Arial', size = 11)
    for  cell in ws[1]:
        cell.fill = hdr_fill
        cell.font = hdr_font
        cell.alignment = Alignment(horizontal = 'center')

    # C2 - Number formats
    cols = region_summary.columns.tolist()
    rev_col   = get_column_letter(cols.index('total_revenue') + 1)
    avg_col   = get_column_letter(cols.index('avg_revenue')   + 1)
    cnt_col   = get_column_letter(cols.index('order_count')   + 1)

    for row in ws.iter_rows(min_row = 2):
        for cell in row:
            col_letter = get_column_letter(cell.column)
            if col_letter == rev_col: cell.number_format = '#,##0.00'
            if col_letter == avg_col: cell.number_format = '#,##0.00'
            if col_letter == cnt_col: cell.number_format = '#,##0'

    # C2 - Alternating rows
    light = PatternFill('solid', fgColor='EBF0F8')
    for i, row in enumerate(ws.iter_rows(min_row = 2), start = 2):
        if i % 2 == 0:
            for cell in row:
                cell.fill = light

    # C3 - Column widths + freeze
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4
    ws.freeze_panes = 'A2'

    print(f"Formatted - rows: {ws.max_row}, cols: {ws.max_column}")

verify = pd.read_excel('ak_shopease_formatted.xlsx', sheet_name = 'Executive Summary')
print(verify.to_string(index = False))

Formatted - rows: 5, cols: 4
region  total_revenue  order_count  avg_revenue
  East      136318.29           47      2900.39
 North      132736.10           57      2328.70
 South      122708.43           52      2359.78
  West       97069.65           44      2206.13


---

### Task D — Export Insight Summary (15 pts)

**Business question:** Every exported file needs a cover note — a brief text summary
a non-technical manager can read before opening the Excel.

#### D1 — Print a Formatted Summary Report (15 pts)
Using only `print()` statements (no file writing), produce a clean text report:

```
============================================================
  SHOPEASE ANALYTICS — EXPORT SUMMARY
  Generated: [today's date]
============================================================

FILES EXPORTED
  1. delivered_orders.csv   — [X] rows, [Y] cols
  2. shopease_report.xlsx   — 3 sheets
  3. shopease_formatted.xlsx — 1 formatted sheet

REGION PERFORMANCE
  [print region_summary as aligned text — use .to_string()]

CATEGORY PERFORMANCE  
  [print category_summary as aligned text]

KEY NUMBERS
  Total Orders    : [X]
  Total Revenue   : ₹[X,XXX,XXX.XX]
  High Value Orders: [X] ([X.X]% of total)
  Return Rate     : [X.X]%

RECOMMENDED ACTION
  [One NRA sentence citing the top region and top category by revenue]
============================================================
```

All numbers must be computed from your DataFrames — no hardcoding.


In [11]:
# D1 — Formatted Text Summary Report (15 pts)
# Use f-strings and print() to produce the full report above
# All values computed from: df, region_summary, category_summary
# No hardcoded numbers

import datetime
total_rev    = df['revenue'].sum()
high_v       = (df['rev_tier']=='High Value').sum()
high_pct     = high_v / len(df) * 100
return_rate  = (df['status']=='Returned').sum() / len(df) * 100
top_region   = region_summary.iloc[0]
top_category = category_summary.iloc[0]

print("=" * 60)
print("  SHOPEASE ANALYTICS — EXPORT SUMMARY")
print(f"  Generated: {datetime.date.today()}")
print("=" * 60)
print("\nFILES EXPORTED")
reimport_shape = pd.read_csv('ak_delivered_orders.csv').shape
print(f"  1. delivered_orders.csv    — {reimport_shape[0]} rows, {reimport_shape[1]} cols")
print(f"  2. shopease_report.xlsx    — 3 sheets")
print(f"  3. shopease_formatted.xlsx — 1 formatted sheet")
print("\nREGION PERFORMANCE")
print(region_summary.to_string(index=False))
print("\nCATEGORY PERFORMANCE")
print(category_summary.to_string(index=False))
print("\nKEY NUMBERS")
print(f"  Total Orders     : {len(df)}")
print(f"  Total Revenue    : ₹{total_rev:,.2f}")
print(f"  High Value Orders: {high_v} ({high_pct:.1f}% of total)")
print(f"  Return Rate      : {return_rate:.1f}%")
print("\nRECOMMENDED ACTION")
print(f"  {top_region['region']} leads revenue at ₹{top_region['total_revenue']:,.2f}; "
      f"{top_category['category']} is the top category at ₹{top_category['total_revenue']:,.2f}. "
      f"Replicate {top_region['region']}'s order mix in lower-performing regions.")
print("=" * 60)



  SHOPEASE ANALYTICS — EXPORT SUMMARY
  Generated: 2026-04-26

FILES EXPORTED
  1. delivered_orders.csv    — 139 rows, 8 cols
  2. shopease_report.xlsx    — 3 sheets
  3. shopease_formatted.xlsx — 1 formatted sheet

REGION PERFORMANCE
region  total_revenue  order_count  avg_revenue
  East      136318.29           47      2900.39
 North      132736.10           57      2328.70
 South      122708.43           52      2359.78
  West       97069.65           44      2206.13

CATEGORY PERFORMANCE
   category  total_revenue  order_count  avg_revenue  high_value_orders
Electronics      143021.83           62      2306.80                  7
   Clothing      123304.83           44      2802.38                  7
       Home       93885.39           35      2682.44                  3
     Sports       85764.71           42      2042.02                  4
      Books       42855.71           17      2520.92                  3

KEY NUMBERS
  Total Orders     : 200
  Total Revenue    : ₹488,832.47


---
## 📊 Section 4 — Scoring Rubric

| Task | Sub-Task | Pts | What Is Checked |
|------|----------|-----|-----------------|
| **A — CSV** | A1 Filtered Export | 8 | Correct filter, 8 cols, index=False, utf-8-sig, read-back shape correct |
| | A2 Verification | 7 | Column list correct, no Unnamed:0, tier value_counts, NRA present |
| **B — Multi-Sheet** | B1 Three DataFrames | 8 | Correct aggs, high_value_orders column, sort order |
| | B2 Excel Write | 12 | 3 sheets written, correct names, read-back shapes all correct |
| **C — Formatted** | C1 Header Format | 10 | Dark blue fill, white bold Arial, center aligned, row/col print |
| | C2 Number + Rows | 10 | 3 number formats correct columns, alternating fill even rows |
| | C3 Widths + Freeze | 10 | Auto-width all cols, freeze at A2, read-back data intact |
| **D — Summary** | D1 Text Report | 15 | All sections present, all numbers computed (not hardcoded), NRA at end |
| **★ Bonus** | See below | 10 | — |
| **TOTAL** | | **80 + 10★** | |

---

### ⭐ Bonus Task — Conditional Formatting: Traffic Light Revenue Column (10★ pts)

In `'shopease_formatted.xlsx'` sheet `'Executive Summary'`, apply **conditional cell colours**
to the `total_revenue` column (data rows only):

- Revenue ≥ `region_summary['total_revenue'].quantile(0.75)` → green fill `'C6EFCE'`, green font `'276221'`
- Revenue between Q25 and Q75 → yellow fill `'FFEB9C'`, dark yellow font `'9C5700'`
- Revenue < Q25 → red fill `'FFC7CE'`, red font `'9C0006'`

This requires loading the saved workbook with `openpyxl.load_workbook()`,
applying fills cell by cell, then saving.

Print: which region got which colour, based on their revenue.


In [12]:
# ⭐ BONUS — Conditional Traffic Light Formatting (10★ pts)
# 1. Load 'shopease_formatted.xlsx' (or build fresh) with load_workbook()
# 2. Find Q25 and Q75 of total_revenue in region_summary
# 3. Loop through total_revenue column data rows
# 4. Apply green / yellow / red fill + font based on thresholds
# 5. Save workbook
# 6. Print which region got which colour

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

# Use the same file you already created in Task C
file_path = 'ak_shopease_formatted.xlsx'
wb = load_workbook(file_path)
ws = wb['Executive Summary']

# Calculate quartiles from region_summary (still in memory)
q1 = region_summary['total_revenue'].quantile(0.25)
q3 = region_summary['total_revenue'].quantile(0.75)

# Define fills and fonts
green_fill = PatternFill('solid', fgColor='C6EFCE')
green_font = Font(color='276221')
yellow_fill = PatternFill('solid', fgColor='FFEB9C')
yellow_font = Font(color='9C5700')
red_fill = PatternFill('solid', fgColor='FFC7CE')
red_font = Font(color='9C0006')

# Find the column index for 'total_revenue' (header row)
header = [cell.value for cell in ws[1]]
col_idx = header.index('total_revenue') + 1  # openpyxl is 1-based

# Apply formatting and collect region→colour mappings
region_color = {}
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    region = row[0].value                # first column = region name
    revenue = row[col_idx - 1].value     # total_revenue cell
    if revenue >= q3:
        fill, font = green_fill, green_font
        color = 'green'
    elif revenue >= q1:
        fill, font = yellow_fill, yellow_font
        color = 'yellow'
    else:
        fill, font = red_fill, red_font
        color = 'red'
    row[col_idx - 1].fill = fill
    row[col_idx - 1].font = font
    region_color[region] = color

# Save the updated workbook
wb.save(file_path)

# Print results
for region, color in region_color.items():
    print(f"{region}: {color}")

East: green
North: yellow
South: yellow
West: red


---

### Key Takeaway
Export is not a technical step — it's a communication step. The data is identical whether
you export with no formatting or full formatting. The difference is whether the client
trusts your work at first glance. Professional export habits (clean CSVs, multi-sheet workbooks,
formatted headers, frozen panes, number formats) take 20 minutes extra and determine whether
you get repeat business.

### Interview Frame
> "I export deliverables in two formats: clean CSV for any downstream pipeline or database,
> and a formatted Excel workbook for the client — dark blue headers, currency number formats,
> frozen panes, and a summary sheet on top. The data is identical, but the formatted file
> is what the client actually reads. I've found that presentation quality is often what
> determines whether analysis gets acted on."

---
